## Setup

### Imports

In [ ]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import colorcet as cc
import yaml
from statsmodels.distributions.copula.api import GumbelCopula

import gumbel_copula_2dRP as rp
import RP_plotting as rp_plot
import regional_declustering as declust
import kendall_interp as kendall


# set up plot preferences
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['font.size'] = 8

### Constants

In [ ]:
def load_config(config_path):
    '''Loads the configuration from a YAML file.'''
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

config = load_config('config/config.yaml')

In [ ]:
# Config parameters
# THRESH = config['THRESH']
THRESH=30
REGIONS = config['REGIONS']
# MODEL = config['MODEL']
MODEL = 'usgs'
T = config['T']
n = config['n']

# Other parameters
# EXPORTS = True
# SAVEFIG = True
EXPORTS = False
SAVEFIG = False

cmap = cc.cm['kbc_r']
u_vals = np.linspace(0.8, 0.999, n)
v_vals = np.linspace(0.8, 0.999, n)

FRAC = 0.2
BUFFER = 5

### Data

In [ ]:
# Drought summary data
if MODEL == 'obs':
    # Obs
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/Drought Data/Drought_Properties_jd.csv',
        parse_dates=['start', 'end', 'previous_end'],
        index_col='StaID'
        )
    
elif MODEL == 'nwm':
    # NWM
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/nwm_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
    drought_data = drought_data[drought_data['pct_type'] == 'weibull_jd_mod']
    
elif MODEL == 'usgs':
    # USGS
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/lstm_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
    drought_data = drought_data[drought_data['pct_type'] == 'weibull_jd_mod']

else:
    # CBRFC
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/cbrfc_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
    drought_data = drought_data[drought_data['pct_type'] == 'weibull_jd_mod']

# List of study gages with HCDN clusters
gages = pd.read_csv(
    '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/CRB Gages/NWM_v3_CRB_with_HCDN_cluster.csv',
    index_col='USGS_ID'
    )[['Lat', 'Lon', 'region']]

# Regions
regions = gages['region'].unique().tolist()
region_names = [
    'Southwest',
    'California and Interior West',
    'Rocky Mountains'
]
# print(regions[REGION])

## Testing

In [ ]:
# # Filter to region
# # These are the local events
# events = drought_data[drought_data['threshold'] == THRESH]
# gages_in_region = gages[gages['region'] == regions[2]].index.tolist()
# events = events[events.index.isin(gages_in_region)]
# events = events[['start', 'end', 'severity', 'duration']]

In [ ]:
# sites = events.index.unique()
# n_sites = len(sites)
# k_min = int(np.ceil(0.2 * n_sites))

# buffer = pd.Timedelta(days=5)

# boundaries = []

# for _, row in events.iterrows():
#     s = row.start - buffer
#     e = row.end + buffer

#     boundaries.append((s, +1))
#     # decrement just after end to keep end inclusive
#     boundaries.append((e + pd.Timedelta(seconds=1), -1))

# boundaries.sort(key=lambda x: x[0])

# # events = []
# # active = 0
# # in_event = False

# # for t, delta in boundaries:
# #     prev_active = active
# #     active += delta

# #     # Start of regional event
# #     if (not in_event) and active >= k_min:
# #         t_start = t
# #         in_event = True

# #     # End of regional event
# #     if in_event and active < k_min:
# #         t_end = t
# #         events.append((t_start, t_end))
# #         in_event = False

In [ ]:
# # Compute the regional events
# events['site'] = events.index

# regional_events = declust.regional_concurrence_intervals(
#     events,
#     frac_thresh = FRAC,
#     buffer = BUFFER,
#     end_gap = BUFFER
#     )

In [ ]:
# # get the duration and severity
# durations, severities = declust.regional_metrics_from_intervals(
#     events,
#     regional_events,
#     severity_method='sum',
#     duration_method='total'
#     )

In [ ]:
# events_reg = pd.DataFrame(
#     {
#         'duration':     durations,
#         'severity':     severities
#     }
# )

## Compare OG and clustered distributions

In [ ]:
# fig, [ax1, ax2] = plt.subplots(figsize=(4,2), nrows=1, ncols=2, dpi=200, layout='constrained')

# events['duration'].plot.kde(
#     ax=ax1,
#     linewidth=1,
#     linestyle='-',
#     color='orange',
#     label=f'All events\nn={len(events)}\nmax duration={max(events['duration']):.0f} days\nmax severity={max(events['severity']):.0f}'
#     )

# # ax.clear()

# events_reg['duration'].plot.kde(
#     ax=ax1,
#     linewidth=1,
#     linestyle='-',
#     color='blue',
#     label=f'Clustered events\nn={len(events_reg)}\nmax duration={max(events_reg['duration']):.0f} days\nmax severity={max(events_reg['severity']):.0f}'
#     )

# ax2.clear()
# events['severity'].plot.kde(
#     ax=ax2,
#     linewidth=1,
#     linestyle='-',
#     color='orange',
#     # label=f'All events\nn={len(events)}\nmax={max(events['severity']):.0f}'
#     label=''
#     )

# events_reg['severity'].plot.kde(
#     ax=ax2,
#     linewidth=1,
#     linestyle='-',
#     color='blue',
#     # label=f'Clustered events\nn={len(events_reg)}\nmax={max(events_reg['severity']):.0f}'
#     label=''
#     )


# # ax1
# ax1.spines[:].set_linewidth(0.5)
# ax1.set_xlabel('Duration (days)')
# ax1.set_ylabel('Density')
# ax1.set_xlim(0)
# ax1.set_ylim(0, 5e-2)
# ax1.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

# # ax2
# ax2.spines[:].set_linewidth(0.5)
# ax2.set_xlabel('Severity')
# ax2.set_xlim(0)
# ax2.set_ylim(0, 5e-3)
# ax2.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))

# # fig.legend(loc='upper right', frameon=False)
# fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.3), ncols=4, frameon=False)


## Fit the Marginals and Copula Function

In [ ]:
out_duration = []   # the most likely duration values
out_severity = []   # the most likely severity values
out_upper_d = []    # CI bounds
out_lower_d = []
out_upper_s = []
out_lower_s = []
out_n_eff = []

delta_t = pd.Timedelta('30D')

fig, axes = plt.subplots(
    figsize=(6.5, 2.5),
    dpi=300,
    nrows=1,
    ncols=3,
    sharex=True,
    sharey=True,
    layout='constrained'
)

for ax, REGION in zip(axes, REGIONS):
    
    ax.spines[:].set_linewidth(0.5)
    ax.grid('major', linewidth=0.5, linestyle=':')
    
    # Filter to region
    events = drought_data[drought_data['threshold'] == THRESH]
    gages_in_region = gages[gages['region'] == regions[REGION]].index.tolist()
    events = events[events.index.isin(gages_in_region)]
    events = events[['start', 'end', 'severity', 'duration']]
    
    # Compute the regional events
    events['site'] = events.index

    regional_events = declust.regional_concurrence_intervals(
        events,
        frac_thresh = FRAC,
        buffer = BUFFER,
        end_gap = BUFFER
        )
        
    D_reg, S_reg = declust.regional_metrics_from_intervals(
        events,
        regional_events,
        severity_method='max',
        duration_method='max'
        )
    
    ax.set_title(f'{region_names[REGION]} (n={len(regional_events)})', fontsize=8)
    
    # Fit EVs
    duration_rv, duration_params, duration_aic = rp.best_fit_rv(D_reg, ['gamma', 'weibull_min', 'expon'], print_out=True)
    severity_rv, severity_params, severity_aic = rp.best_fit_rv(S_reg, ['gamma', 'weibull_min', 'expon'], print_out=True)
    # duration_rv, duration_params, duration_aic = rp.best_fit_rv(D_reg, ['gamma', 'weibull_min'], print_out=True)
    # severity_rv, severity_params, severity_aic = rp.best_fit_rv(S_reg, ['gamma', 'weibull_min'], print_out=True)

    duration_rv = duration_rv(*duration_params)
    severity_rv = severity_rv(*severity_params)
    
    # fit theta
    copula = GumbelCopula()
    theta = copula.fit_corr_param(
        np.vstack((D_reg, S_reg)).transpose()
        )
    
    
    all_duration = []
    all_severity = []
    all_likelihood = []
    max_duration = []       # duration value with max likelihood
    max_severity = []       # severity value with max likelihood
    CI_upper_x = [0]
    CI_upper_y = [0]
    CI_lower_x = [0]
    CI_lower_y =[0]
    
    for t in T:
    
        # compute the iso-line
        V = rp.iso_rp_OR(u_vals, t, theta)
        
            # Kendall contour interpretation
        # k_dom = kendall.dominance_physical(
        #     u_vals,
        #     V,
        #     duration_rv,
        #     severity_rv
        # )
        
        # transform to duration and severity values
        duration_val = duration_rv.ppf(u_vals)
        severity_val = severity_rv.ppf(V)
        all_duration.append(duration_val)
        all_severity.append(severity_val)

        # compute the likelihood along iso-RP
        likelihood = rp.joint_density_OR(u_vals, V, theta, duration_rv, severity_rv)
        # Normalize likelihood to [0, 1], ignoring NaN values
        min_val = np.nanmin(likelihood)
        max_val = np.nanmax(likelihood)
        if max_val != min_val:
            likelihood_normalized = (likelihood - min_val) / (max_val - min_val)
        else:
            likelihood_normalized = np.zeros_like(likelihood)
        
        all_likelihood.append(likelihood_normalized)
        
        # get the duration and severity values at the max likelihood of the iso-line
        ind_max = np.nanargmax(likelihood_normalized)
        max_duration.append(duration_val[ind_max])
        max_severity.append(severity_val[ind_max])


        
        # compute the 95% confidence interval around the max likelihood
        logL = np.log(likelihood)
        logL_max = np.nanmax(logL)
        threshold = logL_max - 0.5 * sp.stats.chi2.ppf(0.95, df=1)
        mask = logL >= threshold
        dur_ci = duration_val[mask]
        sev_ci = severity_val[mask]
        
        CI_upper_x.append(np.min(dur_ci))
        CI_upper_y.append(np.max(sev_ci))
        CI_lower_x.append(np.max(dur_ci))
        CI_lower_y.append(np.min(sev_ci))
    
    # to shade the CI area, need to interpolate to a shared x grid
    x1 = np.array(CI_upper_x)
    y1 = np.array(CI_upper_y)

    x2 = np.array(CI_lower_x)
    y2 = np.array(CI_lower_y)

    # polygon for p_fill
    x_poly = np.concatenate([
        x1,
        [x2[-1]],   # straight line to end of line 2
        x2[::-1],
        [x1[0]]     # straight line back to start of line 1
    ])

    y_poly = np.concatenate([
        y1,
        [y2[-1]],
        y2[::-1],
        [y1[0]]
    ])
    
    # plot the iso-lines colored by likelihood
    for t in range(len(T)):
        lc = rp_plot.colored_line(
            all_duration[t],
            all_severity[t],
            all_likelihood[t],
            cmap=cmap,
            norm=plt.Normalize(0, 1),
            linewidth=2)
        
        ax.add_collection(lc)


    # plot the max likelihood events
    ax.plot(
        max_duration,
        max_severity,
        linestyle='',
        marker='o',
        color='r',
        ms=4,
        label='Most likely event'
    )

    # label each max likelihood event with corresponding return period
    for i, t in enumerate(T):
        ax.text(max_duration[i] * 1.05, max_severity[i] * 1.05, f'{t} yr', fontsize=6, ha='left', va='bottom', color='r')

    # fill the CI region
    ax.fill(
        x_poly,
        y_poly,
        color='silver', #cmap(0.95),
        linewidth=0,
        alpha=0.5,
        label='95% Conf. Int.'
        )


    # proxy object for iso-line legend item
    proxy = mlines.Line2D(
        [], [],
        color=cmap(0.5),
        linewidth=2
    )
    
    out_duration.append(max_duration)
    out_severity.append(max_severity)
    out_upper_d.append(CI_upper_x[1:])
    out_lower_d.append(CI_lower_x[1:])
    out_upper_s.append(CI_upper_y[1:])
    out_lower_s.append(CI_lower_y[1:])

cbar = fig.colorbar(lc, ax=ax, ticks=[0,1], shrink=0.5, aspect=15)
ax.set_ylim(0)
ax.set_xlim(0)
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
fig.supxlabel('Duration (days)', fontsize=8)
fig.supylabel('Severity (percentile \u00D7 days)', fontsize=8)

cbar.ax.spines['outline'].set(visible=True, lw=0.5, edgecolor='black')
cbar.ax.set_yticklabels(['Least likely', 'Most likely'], va='center', rotation=90)

handles, labels = ax.get_legend_handles_labels()
handles.insert(0, proxy)
labels.insert(0, 'Return period')
# ax.legend(handles, labels)
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.1), ncols=3, frameon=False, fontsize=8)

if SAVEFIG:
    plt.savefig(f'/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/Drought Data/RP_Figures/Kendall/Declustered/Mod/Copula_RP_ALL_REGIONS_{MODEL}_{THRESH}.png', dpi=300, bbox_inches='tight')

In [ ]:
# x = duration_rv.ppf(u_vals)
# y = severity_rv.ppf(V)

# dx = np.diff(x)
# dy = np.diff(y)
# ds = np.sqrt(dx**2 + dy**2)

# integrand = 0.5 * ((x[:-1] - y[:-1]) + (x[1:] - y[1:]))

# ds = ds[~np.isnan(ds)]
# integrand = integrand[~np.isnan(integrand)]

# a = np.sum(integrand * ds) / np.sum(ds)

## Export the data

In [ ]:
if EXPORTS:
    df = {
        'Region':       [x for x in region_names for _ in range(5)],
        'T':            T * 3,
        'Duration':     [item for sublist in out_duration for item in sublist],
        'Severity':     [item for sublist in out_severity for item in sublist],
        'D_upper':      [item for sublist in out_upper_d for item in sublist],
        'D_lower':      [item for sublist in out_lower_d for item in sublist],
        'S_upper':      [item for sublist in out_upper_s for item in sublist],
        'S_lower':      [item for sublist in out_lower_s for item in sublist]
    }

    df = pd.DataFrame(df)
    df = df.round(0)
    
    df.to_csv(f'/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/Drought Data/RP_Figures/Kendall/Declustered/Mod/RP_data_{MODEL}_{THRESH}.csv', index=False)